In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json


from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import pickle
from sklearn.impute import KNNImputer
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer

# Modelos
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV
from lightgbm import LGBMRegressor


# Métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



# ------------------- CONFIGURACIÓN ------------------- #

COLUMNAS_MODELO = [
    "superficie_construida", "banos", "distrito_encoded", "habitaciones",
    "planta_numerica", "exterior", "antiguedad", "terraza", "garaje", "calefaccion"
]



# ------------------- CARGAR Y PREPROCESAR ------------------- #

df = pd.read_csv("rental_properties_clustered.csv")
df = df.dropna()

# Encoding por media del precio
df["distrito_encoded"] = df["distrito"].map(df.groupby("distrito")["price_eur_pm"].mean())

# Imputar y escalar todo el dataset para clustering
imputer = SimpleImputer(strategy="mean")
X_imputado = imputer.fit_transform(df[COLUMNAS_MODELO])

scaler_global = StandardScaler()
X_scaled_global = scaler_global.fit_transform(X_imputado)

# Entrenar KMeans
kmeans = KMeans(n_clusters=5, random_state=42, n_init='auto')
df["cluster"] = kmeans.fit_predict(X_scaled_global)

# Guardar
with open("imputer_rental.pkl", "wb") as f:
    pickle.dump(imputer, f)

with open("scaler_global_rental.pkl", "wb") as f:
    pickle.dump(scaler_global, f)

with open("kmeans_rental.pkl", "wb") as f:
    pickle.dump(kmeans, f)



# ------------------- MODELOS A COMPARAR POR CLÚSTER ------------------- #

modelos_disponibles = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "LightGBM": LGBMRegressor(random_state=42)
}

# Resultados de comparativa
resultados_modelos = []

for cluster_id in df["cluster"].unique():
    df_cluster = df[df["cluster"] == cluster_id]
    X = df_cluster[COLUMNAS_MODELO]
    y = df_cluster["price_eur_pm"].values.reshape(-1, 1)

    # Escalado
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y).ravel()

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

    for nombre_modelo, modelo in modelos_disponibles.items():
        modelo.fit(X_train, y_train)
        y_pred_scaled = modelo.predict(X_test).reshape(-1, 1)
        y_pred = scaler_y.inverse_transform(y_pred_scaled)
        y_test_inv = scaler_y.inverse_transform(y_test.reshape(-1, 1))

        resultados_modelos.append({
            "Cluster": cluster_id,
            "Modelo": nombre_modelo,
            "R2": r2_score(y_test_inv, y_pred),
            "MAE": mean_absolute_error(y_test_inv, y_pred),
            "MSE": mean_squared_error(y_test_inv, y_pred)
        })

# DataFrame con resultados de comparativa
df_resultados_modelos = pd.DataFrame(resultados_modelos)
mejores_modelos_por_cluster = df_resultados_modelos.sort_values("R2", ascending=False).groupby("Cluster").first().reset_index()

# Guardar tabla comparativa
df_resultados_modelos.to_pickle("comparativa_modelos_por_cluster_rental.pkl")



# ------------------- ENTRENAR MEJOR MODELO POR CLÚSTER ------------------- #

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

mejores_parametros_por_cluster = {}

for cluster_id in df["cluster"].unique():
    print(f"\n🔍 Hiperparámetros para clúster {cluster_id}...")

    df_cluster = df[df["cluster"] == cluster_id]
    X = df_cluster[COLUMNAS_MODELO]
    y = df_cluster["price_eur_pm"].values.reshape(-1, 1)

    scaler_X = StandardScaler()
    scaler_y = StandardScaler()
    X_scaled = scaler_X.fit_transform(X)
    y_scaled = scaler_y.fit_transform(y).ravel()

    X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

    grid_search = GridSearchCV(
        estimator=RandomForestRegressor(random_state=42),
        param_grid=param_grid,
        scoring='r2',
        cv=5,
        n_jobs=-1,
        verbose=1
    )

    grid_search.fit(X_train, y_train)

    mejores_parametros_por_cluster[cluster_id] = {
        "mejores_parametros": grid_search.best_params_,
        "mejor_score_cv": grid_search.best_score_
    }

    with open(f"mejor_modelo_rf_cluster_{cluster_id}_rental.pkl", "wb") as f:
        pickle.dump(grid_search.best_estimator_, f)
    with open(f"scaler_X_{cluster_id}_rental.pkl", "wb") as f:
        pickle.dump(scaler_X, f)
    with open(f"scaler_y_{cluster_id}_rental.pkl", "wb") as f:
        pickle.dump(scaler_y, f)

# Guardar resultados
pd.DataFrame.from_dict(mejores_parametros_por_cluster, orient='index').to_pickle("mejores_parametros_rf_por_cluster_rental.pkl")



# ------------------- FUNCIÓN DE PREDICCIÓN ------------------- #

def predecir_precio_vivienda(nueva_vivienda: dict) -> float:
    df_nueva = pd.DataFrame([nueva_vivienda])
    df_nueva = df_nueva.reindex(columns=COLUMNAS_MODELO)

    with open("imputer_rental.pkl", "rb") as f:
        imputer = pickle.load(f)
    with open("scaler_global_rental.pkl", "rb") as f:
        scaler_global = pickle.load(f)
    with open("kmeans_rental.pkl", "rb") as f:
        kmeans = pickle.load(f)

    X_imputado = imputer.transform(df_nueva)
    X_scaled_global = scaler_global.transform(X_imputado)
    cluster = kmeans.predict(X_scaled_global)[0]
    print(f"🏷️ Clúster asignado: {cluster}")

    with open(f"mejor_modelo_rf_cluster_{cluster}_rental.pkl", "rb") as f:
        modelo = pickle.load(f)
    with open(f"scaler_X_{cluster}_rental.pkl", "rb") as f:
        scaler_X = pickle.load(f)
    with open(f"scaler_y_{cluster}_rental.pkl", "rb") as f:
        scaler_y = pickle.load(f)

    X_scaled = scaler_X.transform(X_imputado)
    y_pred_scaled = modelo.predict(X_scaled).reshape(-1, 1)
    y_pred = scaler_y.inverse_transform(y_pred_scaled).ravel()[0]

    print(f"✅ Precio estimado: {y_pred:,.2f} EUR")
    return y_pred

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000032 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 146
[LightGBM] [Info] Number of data points in the train set: 159, number of used features: 10
[LightGBM] [Info] Start training from score 0.062926
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -i

c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000097 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 202
[LightGBM] [Info] Number of data points in the train set: 316, number of used features: 9
[LightGBM] [Info] Start training from score 0.005929
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000138 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 379
[LightGBM] [Info] Number of data points in the train set: 500, number of used features: 7
[LightGBM] [Info] Start training from score -0.003213
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000057 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 257
[LightGBM] [Info] Number of data points in the train set: 496, number of used features: 7
[LightGBM] [Info] Start training from score 0.006899
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000046 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 153
[LightGBM] [Info] Number of data points in the train set: 251, number of used features: 10
[LightGBM] [Info] Start training from score -0.029651
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



🔍 Hiperparámetros para clúster 0...
Fitting 5 folds for each of 162 candidates, totalling 810 fits

🔍 Hiperparámetros para clúster 2...
Fitting 5 folds for each of 162 candidates, totalling 810 fits

🔍 Hiperparámetros para clúster 1...
Fitting 5 folds for each of 162 candidates, totalling 810 fits

🔍 Hiperparámetros para clúster 4...
Fitting 5 folds for each of 162 candidates, totalling 810 fits


In [2]:

# ------------------- PREDICCIÓN DEL MODELO ------------------- #

import pandas as pd
import numpy as np
import pickle

# Datos nuevos
nueva_vivienda = {
    "superficie_construida": 120,
    "banos": 2,
    "distrito_encoded": 2953.936170212766,
    "habitaciones": 3,
    "planta_numerica": 2,
    "exterior": 1,
    "antiguedad": 15,
    "terraza": 1,
    "garaje": 1,
    "calefaccion": 1
}

expected_cols = [
    "superficie_construida", "banos", "distrito_encoded", "habitaciones",
    "planta_numerica", "exterior", "antiguedad", "terraza", "garaje", "calefaccion"
]

# Paso 1: Convertir y reindexar
df_nueva = pd.DataFrame([nueva_vivienda])
df_nueva = df_nueva.reindex(columns=expected_cols)

# Paso 2: Cargar transformadores globales
with open("imputer_rental.pkl", "rb") as f:
    imputer = pickle.load(f)
with open("scaler_global_rental.pkl", "rb") as f:
    scaler_global = pickle.load(f)
with open("kmeans_rental.pkl", "rb") as f:
    kmeans = pickle.load(f)

# Paso 3: Imputar y escalar
X_imputado = imputer.transform(df_nueva)
X_scaled_global = scaler_global.transform(X_imputado)

# Paso 4: Asignar clúster
cluster = kmeans.predict(X_scaled_global)[0]
print(f"🏷️ Clúster asignado: {cluster}")

# Paso 5: Cargar modelo y scalers del clúster
with open(f"mejor_modelo_rf_cluster_{cluster}_rental.pkl", "rb") as f:
    modelo = pickle.load(f)
with open(f"scaler_X_{cluster}_rental.pkl", "rb") as f:
    scaler_X = pickle.load(f)
with open(f"scaler_y_{cluster}_rental.pkl", "rb") as f:
    scaler_y = pickle.load(f)

# Paso 6: Escalar con scaler del clúster
X_scaled_cluster = scaler_X.transform(X_imputado)

# Paso 7: Predecir y revertir escala
y_pred_scaled = modelo.predict(X_scaled_cluster).reshape(-1, 1)
precio_final = scaler_y.inverse_transform(y_pred_scaled).ravel()[0]

print(f"✅ Precio estimado: {precio_final:,.2f} EUR")


🏷️ Clúster asignado: 4
✅ Precio estimado: 2,989.27 EUR


c:\Users\Marta\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
